# Paso 1: Generación de Datos Sintéticos

## ¿Qué son los datos sintéticos?

Los **datos sintéticos** son datos generados artificialmente por ordenador que imitan las características de datos reales. En lugar de utilizar información de pacientes reales (lo cual requeriría permisos y plantearía problemas de privacidad), creamos un conjunto de datos ficticio que sigue los mismos patrones estadísticos que encontraríamos en una clínica real.

## ¿Por qué los usamos aquí?

- **Privacidad**: No necesitamos datos de pacientes reales para desarrollar y probar el sistema.
- **Control**: Podemos definir exactamente cuántos deportistas queremos y qué características tendrán.
- **Reproducibilidad**: Con la misma semilla aleatoria, siempre obtenemos exactamente los mismos datos.
- **Desarrollo**: Nos permiten construir y validar el modelo antes de aplicarlo con datos reales.

Este notebook genera el conjunto de datos base que usaremos en todos los pasos siguientes.

In [1]:
import sys
from pathlib import Path

# Añadir el directorio raíz del proyecto al path
proyecto_raiz = Path("..").resolve()
sys.path.insert(0, str(proyecto_raiz))

from src.generador_datos import generar_dataset
from src.variables import VARIABLES, TOTAL_COLUMNAS
import pandas as pd

## Configuración

Aquí puedes ajustar dos parámetros:

| Parámetro | Descripción | Valor por defecto |
|-----------|-------------|-------------------|
| `N_DEPORTISTAS` | Número de deportistas que tendrá el dataset | 500 |
| `SEMILLA` | Número que controla la aleatoriedad (mismo número = mismos datos siempre) | 42 |

**Recomendación para Roberto**: Empieza con los valores por defecto. Una vez que el sistema funcione correctamente, puedes aumentar `N_DEPORTISTAS` para tener más datos de entrenamiento.

In [2]:
# TODO ROBERTO: Puedes cambiar el número de deportistas y la semilla
N_DEPORTISTAS = 500
SEMILLA = 42

df = generar_dataset(n_deportistas=N_DEPORTISTAS, semilla=SEMILLA)
print(f"Dataset generado: {df.shape[0]} deportistas, {df.shape[1]} columnas")

Generando dataset sintético v2.2 con 500 deportistas (semilla=42)...
  [1/5] Bloque contexto...
  [2/5] Bloque fuerza...
  [3/5] Bloque movilidad...
  [4/5] Bloque control...
  [5/5] Inyectando casos frontera...
  Aplicando reglas v2.2 y calculando score de confianza...

Distribución de riesgo:
  bajo            :  49.6 %
  medio           :  27.8 %
  alto            :  20.6 %
  no_concluyente  :   2.0 %

Distribución de confianza:
  alta            :  33.2 %
  media           :  44.4 %
  baja            :  22.4 %

Dataset generado: 500 filas × 38 columnas.

Dataset generado: 500 deportistas, 38 columnas


## Vista previa de los datos

A continuación se muestran los primeros 10 deportistas del dataset. Puedes ver todas las variables que se han generado para cada uno.

In [3]:
df.head(10)

,cuadriceps_der,cuadriceps_izq,isquiotibiales_der,isquiotibiales_izq,gluteo_medio_der,gluteo_medio_izq,rotadores_externos_cadera_der,rotadores_externos_cadera_izq,aductores_cadera_der,aductores_cadera_izq,...,perfil_exigencia_deportiva,historial_lesional,dolor_percibido_nrs,acwr,indice_estres_descanso,riesgo_lesion,score_total,confianza_score,confianza_categoria,reglas_activadas
0,289.7,383.2,165.3,180.5,120.5,109.1,143.9,133.1,194.9,187.4,...,3,2,4,1.2,18.7,bajo,4.0,92.6,alta,—
1,349.4,320.2,232.6,268.0,140.3,138.5,135.3,128.2,191.3,221.6,...,4,1,2,1.1,17.5,bajo,15.5,66.7,media,—
2,175.5,249.4,142.7,151.7,74.9,56.2,90.0,64.8,83.1,111.5,...,3,2,0,1.0,15.7,medio,19.0,63.0,media,—
3,287.6,362.3,229.9,275.6,143.0,156.4,147.8,137.1,183.6,216.3,...,4,9,0,0.9,19.0,medio,16.5,66.7,media,—
4,555.5,454.4,309.1,306.1,158.5,111.5,82.7,107.5,195.5,259.5,...,3,0,1,0.7,17.6,bajo,6.5,81.5,alta,—
5,446.8,421.8,248.1,309.1,221.1,189.0,183.5,183.4,123.4,158.4,...,4,0,0,0.8,16.4,bajo,3.5,88.9,alta,—
6,273.6,272.0,146.4,133.4,95.1,93.1,89.9,68.8,111.4,110.5,...,4,8,2,0.9,11.5,bajo,15.0,63.0,media,—
7,238.0,360.8,167.6,214.8,135.3,150.1,103.7,148.1,183.8,185.2,...,3,4,0,0.8,27.9,bajo,5.0,88.9,alta,—
8,520.4,442.0,290.4,251.1,233.9,229.0,173.0,199.3,202.7,212.9,...,3,7,1,1.1,8.6,bajo,14.0,74.1,media,—
9,470.9,443.9,321.6,321.5,320.0,309.5,230.0,230.0,329.7,340.0,...,2,3,1,1.3,19.0,bajo,7.0,85.2,alta,—


## Distribución de niveles de riesgo

Una de las columnas más importantes del dataset es el **nivel de riesgo de lesión**. Aquí podemos ver cuántos deportistas caen en cada categoría.

In [4]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# Tabla de frecuencias
print("Distribución de niveles de riesgo:")
print("="*40)
distribucion = df["riesgo_lesion"].value_counts().sort_index()
for nivel, cantidad in distribucion.items():
    porcentaje = cantidad / len(df) * 100
    print(f"  {nivel}: {cantidad} deportistas ({porcentaje:.1f}%)")
print("="*40)

# Gráfico de barras
fig, ax = plt.subplots(figsize=(8, 5))

colores = {"Bajo": "#2ecc71", "Medio": "#f39c12", "Alto": "#e74c3c"}
niveles = distribucion.index.tolist()
valores = distribucion.values.tolist()
barras_colores = [colores.get(n, "#3498db") for n in niveles]

barras = ax.bar(niveles, valores, color=barras_colores, edgecolor="white", linewidth=1.5)

# Etiquetas encima de cada barra
for barra, valor in zip(barras, valores):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 5,
        str(valor),
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

ax.set_title("Distribución de niveles de riesgo de lesión", fontsize=14, pad=15)
ax.set_xlabel("Nivel de riesgo", fontsize=12)
ax.set_ylabel("Número de deportistas", fontsize=12)
ax.set_ylim(0, max(valores) * 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
ruta_fig = proyecto_raiz / "figuras" / "distribucion_riesgo.png"
plt.savefig(ruta_fig, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Figura guardada en: {ruta_fig}")

Distribución de niveles de riesgo:
  alto: 103 deportistas (20.6%)
  bajo: 248 deportistas (49.6%)
  medio: 139 deportistas (27.8%)
  no_concluyente: 10 deportistas (2.0%)
Figura guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/distribucion_riesgo.png


## Estadísticas descriptivas

La tabla siguiente muestra un resumen estadístico de todas las variables numéricas del dataset:

- **count**: número de valores disponibles
- **mean**: media (promedio)
- **std**: desviación estándar (dispersión de los datos)
- **min / max**: valores mínimo y máximo
- **25% / 50% / 75%**: percentiles (el 50% es la mediana)

In [5]:
df.describe().round(2)

,cuadriceps_der,cuadriceps_izq,isquiotibiales_der,isquiotibiales_izq,gluteo_medio_der,gluteo_medio_izq,rotadores_externos_cadera_der,rotadores_externos_cadera_izq,aductores_cadera_der,aductores_cadera_izq,...,single_leg_hop_izq,edad,peso_corporal,perfil_exigencia_deportiva,historial_lesional,dolor_percibido_nrs,acwr,indice_estres_descanso,score_total,confianza_score
count,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,...,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00
mean,353.95,356.79,220.59,225.95,177.34,177.23,125.68,123.76,190.74,193.00,...,153.52,29.55,71.18,3.45,2.32,1.09,1.04,15.66,14.45,71.84
std,127.39,125.92,83.75,81.86,65.70,65.33,49.21,49.16,67.48,68.40,...,28.90,8.15,11.77,1.07,2.10,1.40,0.24,6.90,7.72,13.73
min,80.00,80.00,50.00,50.00,40.00,40.00,25.00,25.50,50.00,50.00,...,77.20,18.00,45.00,1.00,0.00,0.00,0.60,0.00,0.00,33.30
25%,261.88,264.45,158.95,170.55,128.50,127.78,90.07,85.55,140.90,138.60,...,135.30,23.00,62.78,3.00,1.00,0.00,0.90,10.98,8.50,63.00
50%,341.30,347.75,205.90,217.05,167.20,166.55,116.85,116.75,180.00,184.40,...,153.40,29.00,70.70,3.00,2.00,1.00,1.00,15.70,13.50,74.10
75%,445.68,443.82,278.52,280.90,222.52,219.82,154.12,154.68,237.38,238.82,...,171.22,35.00,79.12,4.00,3.00,1.00,1.20,20.12,18.50,81.50
max,600.00,600.00,400.00,400.00,320.00,320.00,230.00,230.00,340.00,340.00,...,240.00,58.00,106.10,5.00,10.00,9.00,2.00,36.00,40.50,100.00


## Guardar datos

Ahora vamos a guardar el dataset generado en un archivo CSV. Este archivo será leído automáticamente por los siguientes notebooks, por lo que es importante ejecutar este paso correctamente.

El archivo se guardará en la carpeta `datos/sinteticos/` dentro del proyecto.

In [6]:
ruta_salida = proyecto_raiz / "datos" / "sinteticos" / "dataset_sintetico.csv"
df.to_csv(ruta_salida, index=False, encoding="utf-8")
print(f"Datos guardados en: {ruta_salida}")

Datos guardados en: /Users/__robeerr/Programacion_Local/IntApp v2/datos/sinteticos/dataset_sintetico.csv


## Resumen y siguiente paso

---

**Lo que hemos hecho en este notebook:**

- Generado un dataset sintético con **500 deportistas** y sus variables fisiológicas y de entrenamiento.
- Comprobado que los datos tienen una distribución realista de niveles de riesgo.
- Guardado el dataset en `datos/sinteticos/dataset_sintetico.csv`.

---

**Siguiente paso: ejecuta el notebook `02_exploracion_datos.ipynb`**

En ese notebook exploraremos el dataset en detalle: veremos qué variables están más relacionadas con el riesgo de lesión, detectaremos valores atípicos y prepararemos los datos para el modelo de machine learning.

---

> Si has modificado `N_DEPORTISTAS` o `SEMILLA`, recuerda volver a ejecutar todos los notebooks desde el principio para que los cambios se propaguen correctamente.